In [1]:
import sagemaker

import boto3
from botocore.exceptions import ClientError
import ast

# secret shhhhh
def get_secret():

    secret_name = "gitlab/tcwyu-respository"
    region_name = "us-east-2"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return secret
    
secret = ast.literal_eval(get_secret())


In [2]:
# git config
git_config = {
    'repo': 'https://git.uwaterloo.ca/medical-ai-lab/bertha_experiments.git',
    'username': secret['username'],
    'password': secret['password']
}

hyperparameters = {
    'batch_size': 64, 
    'mortality_days_threshold': 180
}

estimator_args = {
    "base_job_name": "bertha-inference-mortality-uw-dataset",
    "role": "arn:aws:iam::776029861160:role/service-role/AmazonSageMaker-ExecutionRole-20230328T101238",
    "output_path": "s3://himss2023mimiciv/uw_skynet/output",
    "model_uri": "s3://himss2023mimiciv/uw_skynet/output/bertha-finetuning-mortality-uw-dataset-2023-04-06-21-41-01-834/output/model.tar.gz",
    "image_uri": "776029861160.dkr.ecr.us-east-2.amazonaws.com/smile-uw-containers:medair-huggingface-pytorch-training-test",
    "instance_count": 1,
    "instance_type": "ml.g4dn.xlarge",
    "entry_point": "BERTHA_AWS_Inference.py",
    "git_config": git_config,
    "source_dir": '.',
    "hyperparameters": hyperparameters,
}

pytorch_estimator = sagemaker.estimator.Estimator(**estimator_args)

In [3]:
input_data = {
    # "train": "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/hf_dataset_finetune_mortality_prediction_train/train",
    "test": "s3://himss2023mimiciv/uw_skynet/data/mimic-iv_data/ood_data/hf_dataset_finetune_mp_test_with_ids/",
    "tokenizer": "s3://himss2023mimiciv/uw_skynet/data/tokenizer-mimic-iv-icd-final"
}

In [4]:
pytorch_estimator.fit(inputs=input_data)

Cloning into '/tmp/tmph_zdz0at'...
INFO:sagemaker:Creating training-job with name: bertha-finetuning-mortality-uw-dataset-2023-04-06-21-41-01-834


2023-04-06 21:41:05 Starting - Starting the training job...
2023-04-06 21:41:30 Starting - Preparing the instances for training......
2023-04-06 21:42:26 Downloading - Downloading input data...
2023-04-06 21:43:01 Training - Downloading the training image........................
2023-04-06 21:46:52 Training - Training image download completed. Training in progress..bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
sed: can't read changehostname.c: No such file or directory
gcc: error: changehostname.c: No such file or directory
gcc: fatal error: no input files
compilation terminated.
gcc: error: changehostname.o: No such file or directory
ERROR: ld.so: object '/libchangehostname.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
ERROR: ld.so: object '/libchangehostname.so' from LD_PRELOAD cannot be preloaded (cannot open shared object file): ignored.
2023-04-06 21:47:10,445 sagemaker-training-

# Dataset Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import random
random.seed(10)

In [ ]:
# Get df from Athena


In [ ]:
# Create patient - visit number mapping
subjects_admissions = {}
for item in df[['subject_id', 'hadm_id']].values.tolist():
    subjects_admissions.setdefault(item[0], []).append(item[1])

admission_enum = {}
for subject in subjects_admissions:
    # Remove duplicates but preserve order of insertion
    admissions = list(dict.fromkeys(subjects_admissions[subject]))
    for i, hadm_id in enumerate(admissions, start=1):
        admission_enum.setdefault('hadm_id', []).append(hadm_id)
        admission_enum.setdefault('visit_order', []).append(i)

dfv = pd.DataFrame(admission_enum)
dfv

In [ ]:
df = pd.merge(df, dfv)

In [ ]:
# Select columns to keep
df = df[['subject_id', 'visit_order', 'seq_num', 'icd_code', 'days_until_death']]

In [ ]:
# Turn df into mapping
dataset_dict = {}
for (index, subject_id, visit_order, seq_num, icd_code, days) in tqdm(df.itertuples()):
    dataset_dict.setdefault(subject_id, {}).setdefault("icd_code", []).append(icd_code)
    dataset_dict.setdefault(subject_id, {}).setdefault("visit_order", []).append(visit_order)
    dataset_dict.setdefault(subject_id, {}).setdefault("seq_num", []).append(seq_num)
    dataset_dict.setdefault(subject_id, {}).setdefault("days_until_death", []).append(days)

In [ ]:
# import pickle
# with open('/root/data/mimic-iv_data/aws/mimic_icd_data_dict.pkl', 'wb') as f:
#      pickle.dump(dataset_dict, f)

# with open('/root/data/mimic-iv_data/aws/mimic_icd_data_dict_test.pkl', 'wb') as f:
#      pickle.dump(dataset_dict_test, f)

In [ ]:
def process_function(entry, tokenization_params):
    codes = entry['icd_code']
    # Get coding embedding ids
    encoding = tokenizer(codes, **tokenization_params)
    # Add token type ids for visit sequence
    num_codes_to_keep = min(tokenization_params['max_length'] - 2, len(codes))
    encoding['token_type_ids'][1:num_codes_to_keep+1] = entry['visit_order'][:num_codes_to_keep]
    entry.update(encoding)
    return entry

ds = ds.map(lambda x: process_function(x, tokenization_params), num_proc=4, remove_columns=['icd_code', 'visit_order', 'seq_num', 'dod_label'])

In [ ]:
ds_split = ds.train_test_split(test_size=0.1)

In [ ]:
# Save to root data folder
# ds_split.save_to_disk("/root/data/mimic-iv_data/ood/hf_dataset_pretrain")

# Split into 5+ visits

In [ ]:
all_patient_visits = {}
short_patients = set()
long_patients = set()

for patient, visit_id in df[['subject_id', 'hadm_id']].values.tolist():
    all_patient_visits.setdefault(patient, []).append(visit_id)

for patient, visits in all_patient_visits.items():
    if len(set(visits)) >= 5:
        long_patients.update([patient])
    else:
        short_patients.update([patient])

In [ ]:
df = df[["subject_id", "icd_code"]]  # Drop irrelevant data